In [1]:
import os
import subprocess
import re
import numpy as np
import torch
import pickle
import pytorch_lightning as pl
from Bio import SeqIO
import os
os.chdir("/global/scratch/projects/fc_mvslab/predictors/all_predictors/")

import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn import preprocessing
from pickle import dump
import pytorch_lightning as pl
from actpred.models import ActCNNSystem
from actpred.utils import get_threshold, get_stratified_split
import ipyparallel as ipp
import pandas as pd
import time
import glob

2025-08-05 10:26:37.705206: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
ipp.__version__   # check that at least version 7
mycluster = ipp.Cluster(n = int(os.getenv('SLURM_CPUS_ON_NODE')))
mycluster.start_cluster_sync()

c = mycluster.connect_client_sync()
c.wait_for_engines(n = int(os.getenv('SLURM_CPUS_ON_NODE')))
c.ids

Starting 32 engines with <class 'ipyparallel.cluster.launcher.LocalEngineSetLauncher'>
100%|██████████| 32/32 [00:05<00:00,  5.55engine/s]


[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31]

In [3]:
dview = c[:]
dview.block = True

# Load necessary modules on workers
dview.execute('import sys, os')
dview.execute('sys.path.append(os.path.abspath(".."))')
dview.execute('import pandas as pd')
dview.execute('import numpy as np')
dview.execute('import re, subprocess')
dview.execute('import Bio')
dview.execute('from PADDLE import paddle')
dview.execute('from adpred import ADpred as adp')
dview.execute('import torch')

<AsyncResult(execute): finished>

In [4]:
# Load ADHunter model, trained on GCN4 ortholog set
model = torch.load('/global/scratch/projects/fc_mvslab/predictors/adhunter/adhunter.pt')
model.eval()
scaler = pickle.load(open('/global/scratch/projects/fc_mvslab/predictors/adhunter/scaler.pkl', 'rb'))
pl.seed_everything(0)

Global seed set to 0


0

In [5]:
def preprocessing(sequence):
    '''
    Chops the sequence into 40-mer tiles and converts them to integer valued vectors.
    Returns the one-hot-encoded 40-mers as a tensor
    
    sequence: provided sequence
    output_dir: path to output directory
    fasta_name: path to fasta file
    computed: path to already computed .dis file to skip re-running IUPred
    ''' 
    # Chop sequence into 40-mer tiles
    seqs = np.array([sequence[i:i+40] for i in range(len(sequence)-39)], dtype=object)

    # convert seqs to integer valued vectors
    alphabet="ACDEFGHIKLMNPQRSTVWY"
    aa_to_i = {aa:i for i, aa in enumerate(alphabet)}
    i_to_aa = {i:aa for i, aa in enumerate(alphabet)}
    X = np.asarray([[aa_to_i[aa] for aa in x] for x in seqs])

    # Convert matrix to tensor
    return torch.tensor(X)

In [6]:
dview.push( dict(model=model, scaler=scaler, preprocessing=preprocessing) )

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [7]:
def check_recs(recs):
    aa_lst = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']
    
    for r in recs:
        sequence = str(r.seq)
        assert len(sequence) >= 53, 'Protein sequence must be at least 40 amino acids long' 
        assert len(set(sequence) - set(aa_lst)) == 0, 'No non-standard amino acids allowed'

In [8]:
# Need a wrapper function because map() only operates on one argument
def wrapper(r):
    seq = str(r.seq)
    X = preprocessing(seq)
    with torch.no_grad():
        # Run ADHunter
        y_hat = model(X).reshape(-1)
        y_hat = y_hat.detach().numpy()
        
    # Convert model output to interpretable predicted activity values on the scale of the dataset used to train the model
    y_unscaled = scaler.inverse_transform(y_hat.reshape(-1, 1)).reshape(-1)
    
    adhunter_preds = y_unscaled
    adhunter_centers = np.arange(len(adhunter_preds)) + 40/2
    
    return (seq, (adhunter_centers, adhunter_preds))

def run_adhunter(fasta_name, output_dir):
    recs = list(SeqIO.parse(fasta_name, 'fasta'))
    fasta_name_tmp = fasta_name.split('/')[-1].removesuffix('.fasta').removesuffix('.fa')

    check_recs(recs)
    
    # Run a parallel map, executing the wrapper function on indices 0,...,n-1
    lview = c.load_balanced_view()
    # Cause execution on main process to wait while tasks sent to workers finish
    lview.block = True 
    async_result = lview.map_async(wrapper, recs)

    while not async_result.ready():
        print(f"Progress: {async_result.progress}", end='\r', flush=True)
        time.sleep(0.5)
    
    print("Progress: 100.0% - Done.")
    adhunter_out = dict(async_result.get())

    data = []
    
    for r in recs:
        sequence = str(r.seq)
        name = str(r.id)
        adhunter_centers, adhunter_preds = adhunter_out[sequence]
        adhunter_preds = "[" + ",".join(str(x) for x in adhunter_preds) + "]"
        adhunter_centers = "[" + ",".join(str(x) for x in adhunter_centers) + "]"
        data.append([name, sequence, adhunter_centers, adhunter_preds])
    out_df = pd.DataFrame(data, columns=['name', 'sequence', 'adhunter_centers','adhunter_preds'])
    out_df.to_csv(f"{output_dir}/{fasta_name_tmp}_adhunter_preds.csv", encoding='utf-8')

In [9]:
# run_adhunter("/global/scratch/projects/fc_mvslab/OpenProjects/Sanjana/consensus_AD_predictor/data/interpro_uniprot_evid_1_2_split/pt_0.fasta",
#              "/global/scratch/projects/fc_mvslab/OpenProjects/Sanjana/consensus_AD_predictor/output/interpro_uniprot_evid_1_2_dask/")

In [10]:
all_split_inputs = glob.glob("/global/scratch/projects/fc_mvslab/OpenProjects/Sanjana/consensus_AD_predictor/data/interpro_uniprot_evid_1_2_split/*")
all_split_inputs = [_.split("/")[-1].split("_")[1].split(".")[0] for _ in all_split_inputs]
all_split_inputs = set(all_split_inputs)
all_split_inputs

{'0',
 '1',
 '10',
 '100',
 '101',
 '102',
 '103',
 '104',
 '105',
 '106',
 '107',
 '108',
 '109',
 '11',
 '110',
 '111',
 '112',
 '113',
 '114',
 '115',
 '116',
 '117',
 '118',
 '119',
 '12',
 '120',
 '121',
 '122',
 '123',
 '124',
 '125',
 '126',
 '127',
 '128',
 '129',
 '13',
 '130',
 '131',
 '132',
 '133',
 '134',
 '135',
 '136',
 '137',
 '14',
 '15',
 '16',
 '17',
 '18',
 '19',
 '2',
 '20',
 '21',
 '22',
 '23',
 '24',
 '25',
 '26',
 '27',
 '28',
 '29',
 '3',
 '30',
 '31',
 '32',
 '33',
 '34',
 '35',
 '36',
 '37',
 '38',
 '39',
 '4',
 '40',
 '41',
 '42',
 '43',
 '44',
 '45',
 '46',
 '47',
 '48',
 '49',
 '5',
 '50',
 '51',
 '52',
 '53',
 '54',
 '55',
 '56',
 '57',
 '58',
 '59',
 '6',
 '60',
 '61',
 '62',
 '63',
 '64',
 '65',
 '66',
 '67',
 '68',
 '69',
 '7',
 '70',
 '71',
 '72',
 '73',
 '74',
 '75',
 '76',
 '77',
 '78',
 '79',
 '8',
 '80',
 '81',
 '82',
 '83',
 '84',
 '85',
 '86',
 '87',
 '88',
 '89',
 '9',
 '90',
 '91',
 '92',
 '93',
 '94',
 '95',
 '96',
 '97',
 '98',
 '99'}

In [11]:
completed_inputs = glob.glob("/global/scratch/projects/fc_mvslab/OpenProjects/Sanjana/consensus_AD_predictor/output/interpro_uniprot_evid_1_2_dask/*adhunter*")
completed_inputs = [_.split("/")[-1].split("_")[1] for _ in completed_inputs]
completed_inputs = set(completed_inputs)
completed_inputs

{'0',
 '1',
 '10',
 '100',
 '101',
 '102',
 '103',
 '104',
 '105',
 '106',
 '107',
 '108',
 '109',
 '11',
 '110',
 '111',
 '112',
 '113',
 '114',
 '115',
 '116',
 '117',
 '118',
 '119',
 '12',
 '120',
 '121',
 '122',
 '123',
 '124',
 '125',
 '126',
 '127',
 '128',
 '129',
 '13',
 '130',
 '131',
 '132',
 '133',
 '134',
 '135',
 '14',
 '15',
 '16',
 '17',
 '18',
 '19',
 '2',
 '20',
 '21',
 '22',
 '23',
 '24',
 '25',
 '26',
 '27',
 '28',
 '29',
 '3',
 '30',
 '31',
 '32',
 '33',
 '34',
 '35',
 '36',
 '37',
 '38',
 '39',
 '4',
 '40',
 '41',
 '42',
 '43',
 '44',
 '45',
 '46',
 '47',
 '48',
 '49',
 '5',
 '50',
 '51',
 '52',
 '53',
 '54',
 '55',
 '56',
 '57',
 '58',
 '59',
 '6',
 '60',
 '61',
 '62',
 '63',
 '64',
 '65',
 '66',
 '67',
 '68',
 '69',
 '7',
 '70',
 '71',
 '72',
 '73',
 '74',
 '75',
 '76',
 '77',
 '78',
 '79',
 '8',
 '80',
 '81',
 '82',
 '83',
 '84',
 '85',
 '86',
 '87',
 '88',
 '89',
 '9',
 '90',
 '91',
 '92',
 '93',
 '94',
 '95',
 '96',
 '97',
 '98',
 '99'}

In [12]:
uncompleted_inputs = all_split_inputs - completed_inputs
uncompleted_inputs

{'136', '137'}

In [13]:
i = len(completed_inputs)

for input in uncompleted_inputs:
    print(i, input)
    try:
        run_adhunter(
            "/global/scratch/projects/fc_mvslab/OpenProjects/Sanjana/consensus_AD_predictor/data/interpro_uniprot_evid_1_2_split/pt_" + input + ".fasta", 
            "/global/scratch/projects/fc_mvslab/OpenProjects/Sanjana/consensus_AD_predictor/output/interpro_uniprot_evid_1_2_dask/"
        )
    except Exception as e:
        print(f"Error running adhunter for input {input}: {e}")
    i += 1

136 137
Error running adhunter for input 137: 'list' object has no attribute 'ready'
137 136
Error running adhunter for input 136: 'list' object has no attribute 'ready'
